In [100]:
from pyftdi.ftdi import Ftdi
import time

In [101]:
VID= '0x0403'
PID ='0xfaf0'
SERIAL ='26006611'

MGMSG_HW_REQ_INFO = 0x0005
MGMSG_HW_GET_INFO = 0x0006
MOT_MOVE_ABSOLUTE = 0x0453

DEST_USB = 0x50
SRC_HOST = 0x01

In [102]:
def hdr_with_data(msg_id, data, d=DEST_USB, s=SRC_HOST):
    hdr = bytearray(6)
    hdr[0:2] = msg_id.to_bytes(2, 'little')
    hdr[2:4] = len(data).to_bytes(2, 'little')
    hdr[4] = (d | 0x80) & 0xFF   # MSB set => data following
    hdr[5] = s & 0xFF
    return hdr + data

def hdr_only(msg_id, p1=0, p2=0, d=DEST_USB, s=SRC_HOST):
    hdr = bytearray(6)
    hdr[0:2] = msg_id.to_bytes(2, 'little')
    hdr[2] = p1 & 0xFF
    hdr[3] = p2 & 0xFF
    hdr[4] = d & 0xFF
    hdr[5] = s & 0xFF
    return hdr


In [117]:
dev = Ftdi()
# dev.add_custom_product(0x0403,0xfaf0)
dev.open( vendor=0x0403, product=0xfaf0, serial='26006611')
dev.set_baudrate(115200)
dev.set_line_property(8, 1, 'N')

dev.purge_rx_buffer()
dev.purge_tx_buffer()
time.sleep(0.05)


dev.reset()

# Hardware RTS/CTS flow control
dev.set_flowctrl('')

# Assert RTS
dev.set_rts(True)


In [125]:


# build 6-byte header for HW_REQ_INFO
req = bytearray(6)
req[0:2] = MGMSG_HW_REQ_INFO.to_bytes(2, 'little')
req[2] = 0
req[3] = 0
req[4] = 0x50  # dest = USB
req[5] = 0x01  # src = host


print("Sending REQ_INFO...")
print(req.hex())
dev.purge_buffers()
dev.write_data(req)

# read back 6-byte header
hdr = dev.read_data_bytes(6, 1000)
msg_id = int.from_bytes(hdr[0:2], 'little')
if msg_id != MGMSG_HW_GET_INFO:
    print(f"Unexpected header: 0x{msg_id:04x}")

# read 4-byte payload (serial number)
data = dev.read_data_bytes(4, 1000)
serial = int.from_bytes(data, 'little', signed=False)
print(f"Device serial: {serial}")

# --- Send absolute move to 200000 ---
target = 1600000
chan = (1).to_bytes(2, 'little')
pos  = int(target).to_bytes(4, 'little', signed=True)
payload = chan + pos
frame = hdr_with_data(MOT_MOVE_ABSOLUTE, payload)

print("Sending MOVE_ABSOLUTE 2000...")
dev.purge_buffers()
dev.write_data(frame)

time.sleep(5)


# build 6-byte header for HW_REQ_INFO
req = bytearray(6)
req[0:2] = MGMSG_HW_REQ_INFO.to_bytes(2, 'little')
req[2] = 0
req[3] = 0
req[4] = 0x50  # dest = USB
req[5] = 0x01  # src = host


print("Sending REQ_INFO...")
print(req.hex())
dev.purge_buffers()
dev.write_data(req)



# read back 6-byte header
hdr = dev.read_data_bytes(6, 1000)
msg_id = int.from_bytes(hdr[0:2], 'little')
if msg_id != MGMSG_HW_GET_INFO:
    print(f"Unexpected header: 0x{msg_id:04x}")

# read 4-byte payload (serial number)
data = dev.read_data_bytes(4, 1000)
serial = int.from_bytes(data, 'little', signed=False)
print(f"Device serial: {serial}")


# dev.close()

Sending REQ_INFO...
050000005001
Device serial: 26006611
Sending MOVE_ABSOLUTE 2000...
Sending REQ_INFO...
050000005001
Device serial: 26006611
